<a href="https://colab.research.google.com/github/syoon7079/tjwn/blob/main/09_data_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 0) 설정 — 확정 값은 여기서만 바꿉니다
from google.colab import drive
drive.mount('/content/drive')

SAV_PATH = '/content/drive/MyDrive/hdac_data/HN24_ALL.sav'

AGE_MIN, AGE_MAX = 19, 120          # 19세 이상 성인 (80세 이상은 자료에서 80으로 묶임)

EXPOSURE, OUTCOME = 'L_OUT_FQ', 'HE_BMI'
WEIGHT, STRATA, PSU = 'wt_ntr', 'kstrata', 'psu'

# 외식 빈도 3범주 (변수정의표 기준: 5·6·7 저 / 4 중 / 1·2·3 고)
GROUP_MAP = {
    5: '1_저빈도', 6: '1_저빈도', 7: '1_저빈도',
    4: '2_중빈도',
    1: '3_고빈도', 2: '3_고빈도', 3: '3_고빈도',
}

# 공변량: 변수명 → 설명
COVARIATES = {
    'sex': '성별', 'age': '연령', 'region': '지역(시도)',
    'educ': '교육수준(학력)', 'incm': '소득 4분위',
    'BD1_11': '1년간 음주 빈도', 'BS3_1': '현재 일반담배 흡연',
}

# 변수별 '결측으로 볼 원 코드' (변수정의표 기준)
MISSING_CODES = {
    'L_OUT_FQ': [8, 9, 99, 888],
    'BD1_11': [8, 9],
    'BS3_1': [8, 9],
    'educ': [88, 99],     # 주의: educ 8 은 '대학원'(유효값). 결측은 88·99
    'incm': [8, 9],
}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 1) 라이브러리 + 메타데이터만 먼저 읽기
!pip -q install pyreadstat
import numpy as np, pandas as pd, pyreadstat
pd.set_option('display.max_rows', 200, 'display.width', 140)

_, meta0 = pyreadstat.read_sav(SAV_PATH, metadataonly=True)
lower2real = {c.lower(): c for c in meta0.column_names}
real = lambda n: lower2real.get(n.lower())
label_map = dict(zip(meta0.column_names, meta0.column_labels))
print('전체 변수 수:', len(meta0.column_names), '| 행 수(메타):', meta0.number_rows)

전체 변수 수: 813 | 행 수(메타): 6997


#변수확인

In [3]:
# 2) 사용 변수 존재 여부와 라벨 확인
need = [EXPOSURE, OUTCOME, WEIGHT, STRATA, PSU] + list(COVARIATES)
chk = pd.DataFrame([{'요청변수': v, '실제변수명': real(v), '존재': real(v) is not None,
                     '변수라벨': label_map.get(real(v)) if real(v) else None} for v in need])
display(chk)
assert chk['존재'].all(), f"자료에 없는 변수: {chk.loc[~chk['존재'],'요청변수'].tolist()} → 코드북 원변수명 확인"

,요청변수,실제변수명,존재,변수라벨
0,L_OUT_FQ,L_OUT_FQ,True,최근 1년 동안 평균 외식 빈도
1,HE_BMI,HE_BMI,True,체질량지수
2,wt_ntr,wt_ntr,True,영양조사 가중치
3,kstrata,kstrata,True,분산추정 층
4,psu,psu,True,조사구번호
5,sex,sex,True,성별
6,age,age,True,나이
7,region,region,True,시도
8,educ,educ,True,교육수준: 학력
9,incm,incm,True,소득4분위수(개인)


In [4]:
# 3) 원 코드 값 라벨 확인 — 변수정의표의 코드 의미와 일치하는지 눈으로 대조
for v in [EXPOSURE, OUTCOME] + list(COVARIATES):
    print(f'--- {v} ({label_map.get(real(v))}) ---')
    print(meta0.variable_value_labels.get(real(v), '(값 라벨 없음: 연속형)'))

# L_OUT_FQ 의 3범주 배정 코드가 값 라벨에 실제로 있는지 확인
lab = meta0.variable_value_labels.get(real(EXPOSURE), {})
print('\nGROUP_MAP 코드 중 값 라벨에 없는 코드:', [k for k in GROUP_MAP if float(k) not in {float(x) for x in lab}])

--- L_OUT_FQ (최근 1년 동안 평균 외식 빈도) ---
(값 라벨 없음: 연속형)
--- HE_BMI (체질량지수) ---
(값 라벨 없음: 연속형)
--- sex (성별) ---
(값 라벨 없음: 연속형)
--- age (나이) ---
(값 라벨 없음: 연속형)
--- region (시도) ---
(값 라벨 없음: 연속형)
--- educ (교육수준: 학력) ---
(값 라벨 없음: 연속형)
--- incm (소득4분위수(개인)) ---
(값 라벨 없음: 연속형)
--- BD1_11 (1년간 음주빈도) ---
(값 라벨 없음: 연속형)
--- BS3_1 (현재 일반담배(궐련) 흡연 여부) ---
(값 라벨 없음: 연속형)

GROUP_MAP 코드 중 값 라벨에 없는 코드: [5, 6, 7, 4, 1, 2, 3]


In [5]:
# 4) 필요한 변수만 불러오기
use = list(dict.fromkeys([real(v) for v in need]))
df, meta = pyreadstat.read_sav(SAV_PATH, usecols=use)
E, Y, W, S, P, AGE = real(EXPOSURE), real(OUTCOME), real(WEIGHT), real(STRATA), real(PSU), real('age')
print(df.shape)

(6997, 12)


#원 코드별 빈도표

In [6]:
# 5) 노출 L_OUT_FQ 원 코드 빈도표 (비가중 / 가중)
raw = pd.DataFrame({'비가중 N': df[E].value_counts(dropna=False).sort_index(),
                    '가중 합': df.groupby(E, dropna=False)[W].sum().round(0)})
raw['값 라벨'] = [meta.variable_value_labels.get(E, {}).get(k, '') for k in raw.index]
display(raw)

,비가중 N,가중 합,값 라벨
L_OUT_FQ,,,
1.0,246,2271128.0,
2.0,1083,9559372.0,
3.0,1477,12270776.0,
4.0,829,6710080.0,
5.0,1457,10413694.0,
6.0,1199,7431140.0,
7.0,514,2865844.0,
NaN,192,0.0,


In [7]:
# 6) 3범주 재코딩 + 검증 (전체 연령 기준 / 19세 이상 기준 모두)
df['exp_valid'] = df[E].where(~df[E].isin(MISSING_CODES[EXPOSURE]))
df['out_group'] = df['exp_valid'].map(GROUP_MAP)

# 검증 1: 원 코드 1~7 이 모두 한 번씩만 배정
assert sorted(GROUP_MAP) == list(range(1, 8)), f'원 코드 누락/중복: {sorted(GROUP_MAP)}'
# 검증 2: 결측 코드가 아닌데 매핑 안 된 값이 있는지
unmapped = df.loc[df['exp_valid'].notna() & df['out_group'].isna(), E].unique()
print('매핑되지 않은 원 코드:', unmapped)

display(pd.crosstab(df[E], df['out_group'].fillna('결측'), margins=True).rename_axis('원 코드'))

adult = df[AGE].between(AGE_MIN, AGE_MAX)
print('\n[19세 이상] 외식 빈도 집단별 비가중 인원')
display(df.loc[adult, 'out_group'].value_counts(dropna=False).sort_index().rename('비가중 N').to_frame())

매핑되지 않은 원 코드: []


out_group,1_저빈도,2_중빈도,3_고빈도,All
원 코드,,,,
1.0,0,0,246,246
2.0,0,0,1083,1083
3.0,0,0,1477,1477
4.0,0,829,0,829
5.0,1457,0,0,1457
6.0,1199,0,0,1199
7.0,514,0,0,514
All,3170,829,2806,6805



[19세 이상] 외식 빈도 집단별 비가중 인원


,비가중 N
out_group,
1_저빈도,3134
2_중빈도,802
3_고빈도,1923
NaN,174


In [8]:
# 7) 결과변수 HE_BMI 확인
print('HE_BMI 전체 결측:', int(df[Y].isna().sum()), '| 19세 이상 결측:', int(df.loc[adult, Y].isna().sum()))
print(df.loc[adult, Y].describe())
# 이상값 처리 기준을 적용할 경우: 이용지침서 또는 사전 정의한 근거를 적고 여기에 구현 (현재는 적용하지 않음)

HE_BMI 전체 결측: 116 | 19세 이상 결측: 108
count    5925.000000
mean       24.142586
std         3.875169
min        12.073090
25%        21.459866
50%        23.751785
75%        26.340766
max        48.683874
Name: HE_BMI, dtype: float64


#가중치·층·집락 확인

In [9]:
# 8) 복합표본설계 변수 점검 (전체 자료 기준)
print('층 수:', df[S].nunique(), '| PSU 수:', df[P].nunique())
print('결측:', df[[S, P, W]].isna().sum().to_dict(), '| 가중치<=0:', int((df[W] <= 0).sum()))
psu_per_strata = df.groupby(S)[P].nunique()
print('PSU가 1개뿐인 층:', int((psu_per_strata == 1).sum()))
display(psu_per_strata.describe().to_frame('층당 PSU 수'))

층 수: 27 | PSU 수: 192
결측: {'kstrata': 0, 'psu': 0, 'wt_ntr': 192} | 가중치<=0: 0
PSU가 1개뿐인 층: 0


,층당 PSU 수
count,27.000000
mean,7.111111
std,5.963565
min,2.000000
25%,3.500000
50%,6.000000
75%,7.500000
max,25.000000


#분석대상선정 단계표

In [10]:
# 9) 분석 대상 선정 단계표 (행은 지우지 않고 단계별 조건으로 셈)
steps, cur = [], pd.Series(True, index=df.index)
def add(name, cond, reason):
    global cur
    before = int(cur.sum()); cur = cur & cond
    steps.append({'단계': name, '제외 사유': reason, '제외 인원': before - int(cur.sum()), '남은 인원': int(cur.sum())})

steps.append({'단계': '1. 전체 원자료', '제외 사유': '-', '제외 인원': 0, '남은 인원': len(df)})
add('2. 연령 19세 이상', df[AGE].between(AGE_MIN, AGE_MAX), '미성년자(연령 범위 밖)')
add('3. L_OUT_FQ 유효', df['out_group'].notna(), '외식 빈도 결측/비해당/모름·무응답')
add('4. HE_BMI 유효', df[Y].notna(), 'BMI 결측(검진 미참여 등)')
add('5. 설계변수 유효', df[[S, P, W]].notna().all(axis=1) & (df[W] > 0), '가중치·층·집락 결측 또는 0')
flow = pd.DataFrame(steps); display(flow)

df['analysis_sample'] = cur
print('6. 최종 분석 대상자:', int(cur.sum()), '명 (공변량 결측은 제외하지 않음)')

,단계,제외 사유,제외 인원,남은 인원
0,1. 전체 원자료,-,0,6997
1,2. 연령 19세 이상,미성년자(연령 범위 밖),964,6033
2,3. L_OUT_FQ 유효,외식 빈도 결측/비해당/모름·무응답,174,5859
3,4. HE_BMI 유효,BMI 결측(검진 미참여 등),94,5765
4,5. 설계변수 유효,가중치·층·집락 결측 또는 0,0,5765


6. 최종 분석 대상자: 5765 명 (공변량 결측은 제외하지 않음)


In [11]:
# 10) 3단계 제외 사유 세부: 19세 이상 중 L_OUT_FQ 가 무효인 사람의 실제 원 코드
bad = df.loc[adult & df['out_group'].isna(), E]
print('19세 이상 중 L_OUT_FQ 무효 인원:', len(bad))
display(bad.value_counts(dropna=False).rename('인원').to_frame())

19세 이상 중 L_OUT_FQ 무효 인원: 174


,인원
L_OUT_FQ,
NaN,174


In [12]:
# 11) 공변량: 일괄 제외 없이 변수별 결측만 보고 + 빈도표 (최종 분석 대상자 기준)
sub = df[df['analysis_sample']].copy()
rows = []
for v, desc in COVARIATES.items():
    r = real(v)
    sub[r + '_cl'] = sub[r].where(~sub[r].isin(MISSING_CODES.get(v, [])))
    rows.append({'공변량': v, '설명': desc, '결측 수(처리 후)': int(sub[r + '_cl'].isna().sum()),
                 '고유값 수': sub[r + '_cl'].nunique()})
display(pd.DataFrame(rows))

for v, desc in COVARIATES.items():
    r = real(v)
    if v == 'age':
        print(f'\n[{v}] {desc}'); print(sub[r + '_cl'].describe()); continue
    t = sub[r + '_cl'].value_counts(dropna=False).sort_index().rename('인원').to_frame()
    t['값 라벨'] = [meta.variable_value_labels.get(r, {}).get(k, '') for k in t.index]
    print(f'\n[{v}] {desc}'); display(t)

,공변량,설명,결측 수(처리 후),고유값 수
0,sex,성별,0,2
1,age,연령,0,62
2,region,지역(시도),0,17
3,educ,교육수준(학력),148,8
4,incm,소득 4분위,28,4
5,BD1_11,1년간 음주 빈도,706,6
6,BS3_1,현재 일반담배 흡연,3679,3



[sex] 성별


,인원,값 라벨
sex_cl,,
1.0,2474,
2.0,3291,



[age] 연령
count    5765.000000
mean       53.849263
std        17.114871
min        19.000000
25%        41.000000
50%        55.000000
75%        68.000000
max        80.000000
Name: age_cl, dtype: float64

[region] 지역(시도)


,인원,값 라벨
region_cl,,
1.0,1094,
2.0,371,
3.0,256,
4.0,360,
5.0,190,
6.0,158,
7.0,120,
8.0,123,
9.0,1323,



[educ] 교육수준(학력)


... (3 outputs left out)


[BD1_11] 1년간 음주 빈도


,인원,값 라벨
BD1_11_cl,,
1.0,1058,
2.0,1118,
3.0,653,
4.0,1177,
5.0,772,
6.0,281,
NaN,706,



[BS3_1] 현재 일반담배 흡연


,인원,값 라벨
BS3_1_cl,,
1.0,656,
2.0,113,
3.0,1317,
NaN,3679,


In [13]:
# 12) 집단별 기술통계 (최종 분석 대상자, 가중). 표준오차·검정은 복합표본 분석 단계에서 수행
wmean = lambda g: np.average(g[Y], weights=g[W])
res = (sub.groupby('out_group')
         .apply(lambda g: pd.Series({'비가중 N': len(g), '가중 합': g[W].sum(), '가중 평균 BMI': wmean(g)}))
         .round(2))
display(res)

/tmp/ipykernel_3536/2739967540.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({'비가중 N': len(g), '가중 합': g[W].sum(), '가중 평균 BMI': wmean(g)}))


,비가중 N,가중 합,가중 평균 BMI
out_group,,,
1_저빈도,3062.0,19988801.85,23.97
2_중빈도,797.0,6457869.79,24.18
3_고빈도,1906.0,17299636.87,24.70
